In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import numpy as np
import pandas as pd
import pickle
import warnings

from scipy import stats

# Ignore all warnings
warnings.filterwarnings('ignore')

In [22]:
with open('../results_guards_regression.pkl', 'rb') as f:
    results = pickle.load(f)

In [23]:
base_names = ['LIME', 'SHAP', 'CE', 'PCE', 'CE Guarded', 'CE Unguarded', 'PCE Guarded', 'PCE Unguarded']
norm_names = ['', ' Dist', ' Std', ' Abs', ' Var']
setup_names = {'lime_base': 'LIME Base', 'shap_base': 'SHAP Base'}
for i, base in enumerate(['lime', 'shap', 'ce', 'pce', 'ce_guarded', 'ce_unguarded', 'pce_guarded', 'pce_unguarded']):
    for j, setup in enumerate(['']):#['', '_dist', '_std', '_abs', '_var']
        setup_names[base + setup] = base_names[i] + norm_names[j]
setup_names

{'lime_base': 'LIME Base',
 'shap_base': 'SHAP Base',
 'lime': 'LIME',
 'shap': 'SHAP',
 'ce': 'CE',
 'pce': 'PCE',
 'ce_guarded': 'CE Guarded',
 'ce_unguarded': 'CE Unguarded',
 'pce_guarded': 'PCE Guarded',
 'pce_unguarded': 'PCE Unguarded'}

In [24]:
results.keys()
filtered_results = {k: v for k, v in results.items() if k not in ['num_rep', 'test_size',]}
filtered_results.keys()

dict_keys(['abalone', 'anacalt', 'bank8fh', 'bank8fm', 'bank8nh', 'bank8nm', 'comp', 'concreate', 'deltaA', 'deltaE', 'friedm', 'kin8fh', 'kin8fm', 'kin8nh', 'kin8nm', 'laser', 'mg', 'mortage', 'plastic', 'puma8fh', 'puma8fm', 'puma8nh', 'puma8nm'])

In [26]:

stab_data = []
rob_data = []
for key in filtered_results.keys():
    for setup in filtered_results[key]['RF']['rob_timer'].keys():
        stab_data.append([key, setup_names[setup], np.mean(filtered_results[key]['RF']['stab_timer'][setup])])
        rob_data.append([key, setup_names[setup], np.mean(filtered_results[key]['RF']['rob_timer'][setup])])

stab_df = pd.DataFrame(stab_data, columns=['Dataset', 'Setup', 'Value'])
rob_df = pd.DataFrame(rob_data, columns=['Dataset', 'Setup', 'Value'])
stab_df

,Dataset,Setup,Value
0,abalone,LIME Base,0.065040
1,abalone,SHAP Base,0.200513
2,abalone,LIME,0.079421
3,abalone,SHAP,0.278086
4,abalone,CE,0.004079
...,...,...,...
225,puma8nm,PCE,0.006562
226,puma8nm,CE Guarded,0.005746
227,puma8nm,CE Unguarded,0.002466
228,puma8nm,PCE Guarded,0.008000


In [27]:
import pandas as pd
import numpy as np

# Assuming stab_df and rob_df are already defined

# Define the order of the 'Setup' column
setup_order = stab_df['Setup'].unique()  # or specify the order explicitly: ['Setup1', 'Setup2', ...]

# Convert 'Setup' column to categorical with the specified order
stab_df['Setup'] = pd.Categorical(stab_df['Setup'], categories=setup_order, ordered=True)
rob_df['Setup'] = pd.Categorical(rob_df['Setup'], categories=setup_order, ordered=True)

stab_pivot = stab_df.pivot(index='Dataset', columns='Setup', values='Value')
rob_pivot = rob_df.pivot(index='Dataset', columns='Setup', values='Value')

display(stab_pivot) 
display(stab_df.pivot_table(columns='Setup', values=['Value'], aggfunc=[np.mean]))
display(rob_pivot)
display(rob_df.pivot_table(columns='Setup', values=['Value'], aggfunc=[np.mean]))
stab_pivot.to_csv('stab_time.csv')
rob_pivot.to_csv('rob_time.csv')

Setup,LIME Base,SHAP Base,LIME,SHAP,CE,PCE,CE Guarded,CE Unguarded,PCE Guarded,PCE Unguarded
Dataset,,,,,,,,,,
abalone,0.065040,0.200513,0.079421,0.278086,0.004079,0.005901,0.004258,0.001295,0.006003,0.003510
anacalt,0.032525,0.031940,0.045248,0.049300,0.002682,0.003683,0.002561,0.000691,0.004101,0.002092
bank8fh,0.104942,0.262035,0.125076,0.345914,0.004287,0.006951,0.004951,0.001720,0.006205,0.003927
bank8fm,0.098105,0.238498,0.113195,0.313035,0.004173,0.007195,0.004562,0.001801,0.007165,0.003962
bank8nh,0.111829,0.271680,0.129278,0.355813,0.005266,0.006796,0.004776,0.001815,0.007095,0.004399
bank8nm,0.100596,0.258179,0.118271,0.333875,0.004246,0.006453,0.005243,0.001831,0.006570,0.003679
comp,0.088305,0.611087,0.105896,0.778925,0.005589,0.008858,0.006114,0.002064,0.009285,0.005420
concreate,0.061150,0.142325,0.076495,0.218149,0.003407,0.005328,0.003412,0.001075,0.005533,0.003008
deltaA,0.109892,0.051570,0.129286,0.064880,0.002815,0.004299,0.002854,0.001229,0.004729,0.003790


mean                                                               \
Setup LIME Base SHAP Base      LIME      SHAP        CE       PCE CE Guarded   
Value  0.093076   0.21083  0.109391  0.281489  0.003898  0.006163   0.004215   

                                              
Setup CE Unguarded PCE Guarded PCE Unguarded  
Value     0.001628    0.006205      0.003867

Setup,LIME Base,SHAP Base,LIME,SHAP,CE,PCE,CE Guarded,CE Unguarded,PCE Guarded,PCE Unguarded
Dataset,,,,,,,,,,
abalone,0.062666,0.181401,0.075311,0.256528,0.003274,0.005243,0.003335,0.001106,0.005038,0.002915
anacalt,0.032227,0.027427,0.048514,0.045092,0.002440,0.003926,0.002481,0.000665,0.003829,0.001912
bank8fh,0.131982,3.960876,0.127384,0.367467,0.004290,0.007233,0.004136,0.001712,0.007376,0.004026
bank8fm,0.106010,0.235022,0.117020,0.318894,0.004531,0.006285,0.004455,0.002783,0.007151,0.003881
bank8nh,0.108192,0.272174,0.125266,0.355352,0.004287,0.006341,0.004418,0.001701,0.006918,0.003917
bank8nm,0.103383,0.258451,0.120454,0.344326,0.004828,0.006480,0.004909,0.001789,0.006920,0.004943
comp,0.089706,0.617164,0.105465,0.796573,0.005833,0.009163,0.006018,0.002175,0.009960,0.004571
concreate,0.061838,0.146384,0.078808,0.226420,0.003499,0.005088,0.003875,0.001143,0.005642,0.003201
deltaA,0.113618,0.052274,0.124680,0.065016,0.002675,0.003987,0.002602,0.001191,0.004804,0.002895


mean                                                               \
Setup LIME Base SHAP Base      LIME      SHAP        CE       PCE CE Guarded   
Value  0.095271  0.371386  0.111838  0.284183  0.003978  0.006274   0.004201   

                                              
Setup CE Unguarded PCE Guarded PCE Unguarded  
Value     0.001668    0.006481      0.003804

In [28]:
stab = 'Stability'
rob = 'Robustness'

dataset = {}
for key in filtered_results.keys():
    stability = filtered_results[key]['RF']['stability']
    robustness = filtered_results[key]['RF']['robustness']

    performance = {
        stab: {},
        rob: {},
        'Prediction Variance': np.mean(
            [
                np.var(
                    [
                        robustness['predict'][i][j]
                        for i in range(len(robustness['predict']))
                    ]
                )
                for j in range(len(robustness['predict'][0]))
            ]
        ),
    }
    try:
        for setup in stability.keys():
            # Get the most important feature for each instance using the absolute value of the feature weights
            stab_feature = []
            rob_feature = []
            for j in range(len(stability[setup][0])): # number of instances        
                stab_feature.append([np.argmax(np.abs(stability[setup][i][j]['predict'])) for i in range(len(stability[setup]))])
                rob_feature.append([np.argmax(np.abs(robustness[setup][i][j]['predict'])) for i in range(len(robustness[setup]))])
            stab_feature = stats.mode(stab_feature, axis=1)
            rob_feature = stats.mode(rob_feature, axis=1)


            performance[stab][setup_names[setup]] = np.mean([np.var([stability[setup][i][j]['predict'][stab_feature.mode[j]] for i in range(len(stability[setup]))]) for j in range(len(stability[setup][0]))])
            performance[rob][setup_names[setup]] = np.mean([np.var([robustness[setup][i][j]['predict'][rob_feature.mode[j]] for i in range(len(robustness[setup]))]) for j in range(len(robustness[setup][0]))])
    except:
        print(setup, robustness[setup])
        
    dataset[key] = performance

dataset

{'abalone': {'Stability': {'LIME Base': 8.239784003441834e-06,
   'SHAP Base': 9.14816723583937e-36,
   'LIME': 8.055769344191291e-06,
   'SHAP': 1.9018558200823955e-35,
   'CE': 5.585196838722984e-35,
   'PCE': 1.5407439555097887e-35,
   'CE Guarded': 5.585196838722984e-35,
   'CE Unguarded': 0.0,
   'PCE Guarded': 1.5407439555097887e-35,
   'PCE Unguarded': 0.0},
  'Robustness': {'LIME Base': 0.0008339648233842419,
   'SHAP Base': 0.00014012043723724918,
   'LIME': 0.0009112083137154322,
   'SHAP': 0.00014012043723724918,
   'CE': 0.00235845426303855,
   'PCE': 0.01889238291852353,
   'CE Guarded': 0.00235845426303855,
   'CE Unguarded': 1.4890306122449605e-05,
   'PCE Guarded': 0.01889238291852353,
   'PCE Unguarded': 0.004081723904766137},
  'Prediction Variance': 7.971418367346933e-05},
 'anacalt': {'Stability': {'LIME Base': 2.0864669787353068e-05,
   'SHAP Base': 1.6370404527291504e-34,
   'LIME': 2.0085542843751156e-05,
   'SHAP': 1.6370404527291504e-34,
   'CE': 3.081487911019

In [29]:
stab_data = []
rob_data = []
for key in dataset.keys():
    for setup in dataset[key][stab].keys():
        stab_data.append([key, setup, dataset[key][stab][setup]])
        rob_data.append([key, setup, dataset[key][rob][setup]])
    rob_data.append([key, 'Prediction Variance', dataset[key]['Prediction Variance']])

stab_df = pd.DataFrame(stab_data, columns=['Dataset', 'Setup', 'Value'])
rob_df = pd.DataFrame(rob_data, columns=['Dataset', 'Setup', 'Value'])
display(rob_df)

,Dataset,Setup,Value
0,abalone,LIME Base,0.000834
1,abalone,SHAP Base,0.000140
2,abalone,LIME,0.000911
3,abalone,SHAP,0.000140
4,abalone,CE,0.002358
...,...,...,...
248,puma8nm,CE Guarded,0.004376
249,puma8nm,CE Unguarded,0.000090
250,puma8nm,PCE Guarded,0.013557
251,puma8nm,PCE Unguarded,0.007397


In [30]:
import pandas as pd
import numpy as np

# Assuming stab_df and rob_df are already defined

# Convert 'Setup' column to categorical with the specified order
stab_df['Setup'] = pd.Categorical(stab_df['Setup'], categories=stab_df['Setup'].dropna().unique(), ordered=True)
rob_df['Setup'] = pd.Categorical(rob_df['Setup'], categories=rob_df['Setup'].dropna().unique(), ordered=True)

stab_pivot = stab_df.pivot(index='Dataset', columns='Setup', values='Value')
rob_pivot = rob_df.pivot(index='Dataset', columns='Setup', values='Value')

display(stab_pivot) 
display(stab_df.pivot_table(columns='Setup', values=['Value'], aggfunc=[np.mean]))
display(rob_pivot)
display(rob_df.pivot_table(columns='Setup', values=['Value'], aggfunc=[np.mean]))
stab_pivot.to_csv('stability.csv')
rob_pivot.to_csv('robustness.csv')

Setup,LIME Base,SHAP Base,LIME,SHAP,CE,PCE,CE Guarded,CE Unguarded,PCE Guarded,PCE Unguarded
Dataset,,,,,,,,,,
abalone,0.000008,9.148167e-36,0.000008,1.901856e-35,5.585197e-35,1.540744e-35,5.585197e-35,0.000000e+00,1.540744e-35,0.000000e+00
anacalt,0.000021,1.637040e-34,0.000020,1.637040e-34,3.081488e-35,2.465190e-33,3.081488e-35,0.000000e+00,2.465190e-33,3.274081e-35
bank8fh,0.000013,7.763905e-35,0.000013,9.388908e-36,6.355569e-35,3.697785e-34,6.355569e-35,0.000000e+00,3.697785e-34,1.309632e-34
bank8fm,0.000018,6.608347e-35,0.000017,4.155796e-35,8.474092e-35,1.540744e-34,8.474092e-35,0.000000e+00,1.540744e-34,3.697785e-34
bank8nh,0.000006,1.528707e-35,0.000007,4.273157e-35,4.622232e-35,1.001484e-34,4.622232e-35,0.000000e+00,1.001484e-34,0.000000e+00
bank8nm,0.000009,3.779638e-35,0.000009,8.425944e-36,5.777790e-35,3.081488e-35,5.777790e-35,8.125017e-37,3.081488e-35,0.000000e+00
comp,0.000013,2.420269e-07,0.000012,2.420269e-07,1.232595e-34,3.851860e-35,1.232595e-34,0.000000e+00,3.851860e-35,5.717605e-37
concreate,0.000016,2.214819e-35,0.000016,1.444447e-35,1.232595e-34,6.162976e-34,1.232595e-34,0.000000e+00,6.162976e-34,1.463707e-34
deltaA,0.000002,3.250007e-36,0.000002,3.611119e-36,1.925930e-35,1.140151e-33,1.925930e-35,0.000000e+00,1.140151e-33,1.386670e-34


mean                                                      \
Setup LIME Base     SHAP Base      LIME          SHAP            CE   
Value  0.000016  3.323903e-08  0.000014  3.323903e-08  1.405929e-34   

                                                                             
Setup           PCE    CE Guarded  CE Unguarded   PCE Guarded PCE Unguarded  
Value  8.144237e-34  1.405929e-34  5.626018e-38  8.144237e-34  1.075368e-34

Setup,LIME Base,SHAP Base,LIME,SHAP,CE,PCE,CE Guarded,CE Unguarded,PCE Guarded,PCE Unguarded,Prediction Variance
Dataset,,,,,,,,,,,
abalone,0.000834,0.000140,0.000911,0.000140,0.002358,0.018892,0.002358,1.489031e-05,0.018892,0.004082,0.000080
anacalt,0.006777,0.000183,0.006777,0.000183,0.017872,0.049095,0.017872,0.000000e+00,0.049095,0.006727,0.000046
bank8fh,0.000703,0.000086,0.000672,0.000086,0.006506,0.015639,0.006506,2.681873e-05,0.015639,0.002768,0.000084
bank8fm,0.001201,0.000084,0.001191,0.000084,0.002893,0.018644,0.002893,3.242339e-05,0.018644,0.001923,0.000022
bank8nh,0.000284,0.000038,0.000289,0.000038,0.007361,0.016719,0.007361,4.300631e-05,0.016719,0.015255,0.000132
bank8nm,0.000392,0.000053,0.000402,0.000053,0.002536,0.004491,0.002536,5.891994e-07,0.004491,0.000731,0.000028
comp,0.004689,0.000095,0.004686,0.000095,0.004055,0.030114,0.004055,6.267054e-05,0.030114,0.001046,0.000010
concreate,0.000231,0.000110,0.000238,0.000110,0.006251,0.029748,0.006251,1.366133e-06,0.029748,0.007188,0.000343
deltaA,0.000059,0.000010,0.000060,0.000010,0.000117,0.017855,0.000117,9.370471e-06,0.017855,0.009654,0.000022


mean                                                               \
Setup LIME Base SHAP Base      LIME      SHAP        CE       PCE CE Guarded   
Value  0.001173  0.000238  0.001175  0.000238  0.007918  0.023262   0.007918   

                                                                  
Setup CE Unguarded PCE Guarded PCE Unguarded Prediction Variance  
Value     0.000029    0.023262       0.00683            0.000159